In [1]:
# 1. repo principale
!rm -rf rfi-detection-radioastronomy
!git clone https://github.com/y99dr4s1ll/rfi-detection-radioastronomy.git
%cd rfi-detection-radioastronomy

# 2. RFI-NLN
%cd /kaggle/working
!git clone https://github.com/mesarcik/RFI-NLN.git
%cd rfi-detection-radioastronomy

!pip install faiss-gpu
!pip install aoflagger
!pip install tf-keras

Cloning into 'rfi-detection-radioastronomy'...
remote: Enumerating objects: 269, done.
remote: Counting objects: 100% (269/269), done.
remote: Compressing objects: 100% (192/192), done.
remote: Total 269 (delta 151), reused 185 (delta 67), pack-reused 0 (from 0)
Receiving objects: 100% (269/269), 54.26 KiB | 1.64 MiB/s, done.
Resolving deltas: 100% (151/151), done.
/kaggle/working/rfi-detection-radioastronomy
/kaggle/working
Cloning into 'RFI-NLN'...
remote: Enumerating objects: 1028, done.
remote: Counting objects: 100% (181/181), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 1028 (delta 123), reused 123 (delta 115), pack-reused 847 (from 1)
Receiving objects: 100% (1028/1028), 210.90 KiB | 1.87 MiB/s, done.
Resolving deltas: 100% (731/731), done.
/kaggle/working/rfi-detection-radioastronomy
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 MB 19.7 MB/s eta 0:00:0000:01:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 27.6 MB/s eta 0:00:0000:01

# CUSUM

In [2]:
import sys
sys.path.insert(0, '/kaggle/working/rfi-detection-radioastronomy/src')
sys.path.insert(1, '/kaggle/working/RFI-NLN')
from loaders.luserna_loader import load_luserna, load_luserna_truth
from loaders.lofar_loader import load_lofar

import argparse

2026-03-10 17:42:18.887725: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-10 17:42:18.887825: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-10 17:42:19.001733: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/kaggle/working/RFI-NLN/utils/data/augmentation.py:66: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert(images.shape[1] *0.9 > crop_size[0], ValueError,
/kaggle/working/RFI-NLN/utils/data/augmentation.py:68: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert(images.shape[2]*0.9 > crop_size[1], ValueError,


In [ ]:
import yaml

with open('experiments/configs/cusum_lofar.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['data_path'] = '/kaggle/input/datasets/y99dr4s1ll/lofar-full-rfi-dataset-pkl'

with open('experiments/configs/cusum_lofar.yaml', 'w') as f:
    yaml.dump(cfg, f)

!python experiments/run_cusum_lofar.py

# SUMTHRESHOLD

In [ ]:
!python experiments/run_sum_threshold_lofar.py \
    --rfinln_path /kaggle/working/RFI-NLN \
    --data_path /kaggle/input/datasets/y99dr4s1ll/lofar-full-rfi-dataset-pkl/ \
    --iterations 4 \
    --molt 9

2026-03-10 17:42:46.656413: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-10 17:42:46.656475: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-10 17:42:46.657795: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
Loading LOFAR data...
/kaggle/input/datasets/y99dr4s1ll/lofar-full-rfi-dataset-pkl/LOFAR_Full_RFI_dataset.pkl Loading


# UNET

In [2]:
!python experiments/run_dl.py \
    --config experiments/configs/unet_lofar.yaml \
    --rfinln_path /kaggle/working/RFI-NLN \
    --data_path /kaggle/input/datasets/y99dr4s1ll/lofar-full-rfi-dataset-pkl/

2026-03-10 17:45:31.640509: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-10 17:45:31.640614: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-10 17:45:31.757875: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/kaggle/working/RFI-NLN/utils/data/augmentation.py:66: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert(images.shape[1] *0.9 > crop_size[0], ValueError,
/kaggle/working/RFI-NLN/utils/data/augmentation.py:68: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert(images.shape[2]*0.9 > crop_size[1], ValueError,
Lo

# RNET

In [ ]:
!python experiments/run_dl.py \
    --config experiments/configs/rnet_luserna.yaml \
    --rfinln_path /kaggle/working/RFI-NLN \
    --data_path /kaggle/input/<nome-dataset-spettrogramma> \
    --truth_path /kaggle/input/<nome-dataset-truth>

# NLN

In [2]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

In [3]:
!TF_USE_LEGACY_KERAS=1 python experiments/run_dl.py \
    --config experiments/configs/nln_lofar.yaml \
    --rfinln_path /kaggle/working/RFI-NLN \
    --data_path /kaggle/input/datasets/y99dr4s1ll/lofar-full-rfi-dataset-pkl/

2026-05-05 19:17:51.538163: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-05 19:17:51.538306: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-05 19:17:51.664885: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
/kaggle/working/RFI-NLN/utils/data/augmentation.py:66: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert(images.shape[1] *0.9 > crop_size[0], ValueError,
/kaggle/working/RFI-NLN/utils/data/augmentation.py:68: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert(images.shape[2]*0.9 > crop_size[1], ValueError,
Lo